# Trabalho 2 — Validação, estatística e fairness
## Censo 2022 · API de Agregados IBGE (SIDRA)

**Aluno:** Andre Cardoso de Oliveira  
**Matéria:** Testes Automatizados com IA  
[Tabela 10291](https://sidra.ibge.gov.br/tabela/10291) · [API v3](https://servicodados.ibge.gov.br/api/docs/agregados?versao=3)

Os quatro entregáveis estão nas seções abaixo (sumário do Colab).


In [ ]:
!pip install -q "ipytest==0.14.*" "pandas>=2" "numpy>=1.24"

import gzip, json, urllib.request
import ipytest, numpy as np, pandas as pd, pytest
ipytest.autoconfig()


## Dados

Agregado UF × sexo × cor/raça (ocupados 14+, 2022). Amostra de 8 mil pessoas, proporcional a `n_pessoas`.

`y_true`: mediana do cruzamento ≥ mediana Brasil. **model_x**: UF acima do limiar. **model_y**: classe majoritária.


In [ ]:
AGREGADO, PERIODO, N_AMOSTRA, SEED = 10291, 2022, 8000, 42
BASE = "https://servicodados.ibge.gov.br/api/v3/agregados"
CLASSIF_UF = "2[4,5]|86[2776,2777,2778,2779,2780]|58[82072]"
CLASSIF_BR = "2[6794]|86[95251]|58[82072]"


def ibge_get(q):
    req = urllib.request.Request(f"{BASE}/{q}", headers={"User-Agent": "Trabalho2-PUCMinas"})
    with urllib.request.urlopen(req, timeout=60) as r:
        raw = r.read()
    if raw[:2] == b"\x1f\x8b":
        raw = gzip.decompress(raw)
    return json.loads(raw.decode())


def flatten(payload):
    cells = {}
    for bloco in payload:
        vid = str(bloco["id"])
        for res in bloco["resultados"]:
            cats = {c["nome"]: next(iter(c["categoria"].values())) for c in res["classificacoes"]}
            for s in res["series"]:
                loc = s["localidade"]
                key = (loc["id"], loc["nome"], cats["Sexo"], cats["Cor ou raça"])
                row = cells.setdefault(key, {"uf_id": loc["id"], "uf": loc["nome"], "sexo": cats["Sexo"], "cor_raca": cats["Cor ou raça"]})
                v = s["serie"].get(str(PERIODO))
                row[vid] = np.nan if v in (None, "-", "...", "..", "X") else float(v)
    return pd.DataFrame(cells.values())


LIMIAR = float(ibge_get(f"{AGREGADO}/periodos/{PERIODO}/variaveis/845?localidades=N1[all]&classificacao={CLASSIF_BR}")[0]["resultados"][0]["series"][0]["serie"][str(PERIODO)])
agregado = flatten(ibge_get(f"{AGREGADO}/periodos/{PERIODO}/variaveis/844|845|846?localidades=N3[all]&classificacao={CLASSIF_UF}"))
agregado = agregado.rename(columns={"844": "n_pessoas", "845": "renda_mediana", "846": "renda_media"})
agregado["n_pessoas"] = agregado["n_pessoas"].astype("int64")
agregado["renda_mediana"] = agregado["renda_mediana"].astype("float64")
agregado["renda_media"] = agregado["renda_media"].astype("float64")
for c in ("uf_id", "uf", "sexo", "cor_raca"):
    agregado[c] = agregado[c].astype(object)

rng = np.random.default_rng(SEED)
p = agregado["n_pessoas"].to_numpy(float); p /= p.sum()
people = agregado.iloc[rng.choice(len(agregado), N_AMOSTRA, replace=True, p=p)].reset_index(drop=True)
people["pessoa_id"] = np.arange(1, N_AMOSTRA + 1, dtype=np.int64)
people["y_true"] = (people["renda_mediana"] >= LIMIAR).astype(int)
uf_med = {uf: float(np.average(g["renda_mediana"], weights=g["n_pessoas"])) for uf, g in agregado.groupby("uf")}
people["model_x"] = people["uf"].map(lambda u: int(uf_med[u] >= LIMIAR))
people["model_y"] = int(people["y_true"].mean() >= 0.5)
y_true, group = people["y_true"].to_numpy(), people["sexo"].to_numpy()
model_x_pred, model_y_pred = people["model_x"].to_numpy(), people["model_y"].to_numpy()
print(f"Limiar Brasil R$ {LIMIAR:.0f}  |  {len(agregado)} cruzamentos  |  amostra {len(people)}")
people[["uf", "sexo", "renda_mediana", "y_true", "model_x", "model_y"]].head()


In [ ]:
EXPECTED = {"uf_id": "object", "uf": "object", "sexo": "object", "cor_raca": "object",
            "n_pessoas": "int64", "renda_mediana": "float64", "renda_media": "float64"}


def validate_schema(df, expected_columns=EXPECTED):
    missing = set(expected_columns) - set(df.columns)
    if missing:
        raise ValueError(f"Colunas faltando: {missing}")
    for col, dtype in expected_columns.items():
        if str(df[col].dtype) != dtype:
            raise ValueError(f"Coluna {col} tem tipo {df[col].dtype}, esperado {dtype}")
    return True


def check_no_missing_values(df, columns):
    for col in columns:
        if df[col].isna().any():
            raise ValueError(f"Coluna {col} tem valores ausentes")
    return True


def check_score_range(df, column="renda_mediana", lo=0.0, hi=1_000_000.0):
    if not df[column].dropna().between(lo, hi).all():
        raise ValueError(f"Coluna {column} tem valores fora do intervalo [{lo}, {hi}]")
    return True


def check_unique_ids(df, id_column="pessoa_id"):
    if df[id_column].duplicated().any():
        raise ValueError(f"IDs duplicados em {id_column}")
    return True


def bootstrap_ci(values, n_boot=1000, ci=0.95, seed=42):
    rng = np.random.default_rng(seed)
    values = np.asarray(values, float)
    stats = [rng.choice(values, len(values), replace=True).mean() for _ in range(n_boot)]
    return float(np.percentile(stats, (1 - ci) / 2 * 100)), float(np.percentile(stats, (1 + ci) / 2 * 100))


def bootstrap_ci_diff(a, b, n_boot=1000, ci=0.95, seed=42):
    rng = np.random.default_rng(seed)
    a, b = np.array(a), np.array(b)
    diffs = [rng.choice(a, len(a), replace=True).mean() - rng.choice(b, len(b), replace=True).mean() for _ in range(n_boot)]
    return np.percentile(diffs, (1 - ci) / 2 * 100), np.percentile(diffs, (1 + ci) / 2 * 100)


def compare_models(y_true, pred_a, pred_b):
    lo, hi = bootstrap_ci_diff((np.array(y_true) == pred_a).astype(float), (np.array(y_true) == pred_b).astype(float))
    return lo > 0, lo, hi


def positive_rate(y_pred, group):
    y_pred, group = np.asarray(y_pred), np.asarray(group)
    return {g: float(np.mean(y_pred[group == g])) for g in pd.unique(group)}


def check_demographic_parity(y_pred, group, max_gap=0.1):
    rates = positive_rate(y_pred, group)
    gap = max(rates.values()) - min(rates.values())
    if gap > max_gap:
        raise ValueError(f"Gap de paridade demográfica {gap:.2f} > {max_gap}...")
    return True


def check_accuracy_parity(y_true, y_pred, group, max_gap=0.1):
    y_true, y_pred, group = map(np.asarray, (y_true, y_pred, group))
    accs = {g: float(np.mean(y_true[group == g] == y_pred[group == g])) for g in pd.unique(group)}
    gap = max(accs.values()) - min(accs.values())
    if gap > max_gap:
        raise ValueError(f"Gap de paridade de acurácia {gap:.2f} > {max_gap}...")
    return True


def chave(frame):
    return frame["uf_id"].astype(str) + "|" + frame["sexo"].astype(str) + "|" + frame["cor_raca"].astype(str)


## Entregável 1 — Validação de dados

Schema, completude, faixa e unicidade no agregado SIDRA. O Censo passa; a cópia quebrada falha.


In [ ]:
%%ipytest -v

def test_schema():
    assert validate_schema(agregado) is True
    with pytest.raises(ValueError, match="Colunas faltando"):
        validate_schema(agregado.drop(columns=["sexo"]))


def test_completude():
    assert check_no_missing_values(agregado, list(EXPECTED)) is True
    broken = agregado.copy(); broken.loc[0, "sexo"] = None
    with pytest.raises(ValueError, match="valores ausentes"):
        check_no_missing_values(broken, ["sexo"])


def test_faixa():
    assert check_score_range(agregado, column="renda_mediana", lo=0.0) is True
    broken = agregado.copy(); broken.loc[0, "renda_mediana"] = -1
    with pytest.raises(ValueError, match="fora do intervalo"):
        check_score_range(broken, column="renda_mediana", lo=0.0)


def test_unicidade():
    tmp = agregado.copy(); tmp["chave"] = chave(tmp)
    assert check_unique_ids(tmp, id_column="chave") is True
    dup = pd.concat([tmp, tmp.iloc[[0]]], ignore_index=True)
    with pytest.raises(ValueError, match="duplicados"):
        check_unique_ids(dup, id_column="chave")


## Entregável 2 — Teste estatístico

IC bootstrap 95% da acurácia do `model_x` e comparação com o `model_y`. Diferença significativa se o IC **não contém zero**.


In [ ]:
acerto_x = (y_true == model_x_pred).astype(float)
lo_acc, hi_acc = bootstrap_ci(acerto_x)
melhor, lo_d, hi_d = compare_models(y_true, model_x_pred, model_y_pred)
print(f"acc x {acerto_x.mean():.1%}  IC 95% [{lo_acc:.1%}, {hi_acc:.1%}]")
print(f"acc y {(y_true == model_y_pred).mean():.1%}")
print(f"diff x−y  IC 95% [{lo_d:+.1%}, {hi_d:+.1%}]  significativo={melhor}")


In [ ]:
%%ipytest -v

def test_ic_e_comparacao():
    acerto = (y_true == model_x_pred).astype(float)
    lo, hi = bootstrap_ci(acerto)
    assert lo < acerto.mean() < hi
    melhor, lo_d, hi_d = compare_models(y_true, model_x_pred, model_y_pred)
    assert melhor and lo_d > 0 < hi_d


## Entregável 3 — Fairness

Atributo sensível: **sexo**. Métrica: paridade demográfica (`max_gap = 0,10`) — taxa de “rendimento alto” previsto. Paridade de acurácia só como contraste.


In [ ]:
print("P(y=1)   ", {k: f"{v:.1%}" for k, v in positive_rate(y_true, group).items()})
print("P(pred=1)", {k: f"{v:.1%}" for k, v in positive_rate(model_x_pred, group).items()})


In [ ]:
%%ipytest -v

def test_fairness():
    assert check_demographic_parity(model_x_pred, group) is True
    with pytest.raises(ValueError, match="paridade de acurácia"):
        check_accuracy_parity(y_true, model_x_pred, group)


## Entregável 4 — Interpretação

O rótulo “rendimento alto” já é desigual no IBGE (~62% homens vs ~37% mulheres). Isso é a tabela, não o modelo.

O recorte por **UF** acerta ~85% (IC estreito) e ganha da maioria (~51%) com IC da diferença acima de zero: não é ruído. Como marca o estado inteiro, a taxa de positivos *previstos* quase não muda entre sexos (gap ~0,02) — paridade demográfica **passa**. O acerto, não (~92% vs ~75%) — o erro cai nas mulheres das UFs “ricas”.

Acurácia alta e aprovação parecida não limpam o classificador. As duas métricas juntas descrevem o resultado. A amostra herda a mediana do cruzamento; não é microdado.
